# AI series template

**How to use:** File → Save As → `dayNN_prompt_ai.ipynb`, edit cells 1, 3 and 4, then Run All.

**How the maps link up:** every AI map starts from the same **master table** (all data centers + AI campuses) and the same three base layers, so the look and the cast never change.
Each map adds **one** new thing (water, power, heat, risk...), and saves its new columns back into the master table with `ai.update_master(...)`.
Over the month, the master table becomes the consolidated study.

This template is set up as **Day 1 · Points · Study A**: the base map on its own.

## 1 · Settings

In [1]:
DAY      = 1
PROMPT   = "Points"
STUDY    = "A"            # A Where the Cloud Lands · B What the Cloud Drinks · C What the Cloud Burns · D The Cloud at Risk
SLUG     = "points-ai"
NOTEBOOK = "_template_ai.ipynb"              # this file's name (for the code link in the caption)

TITLE    = "Where the Cloud Lands"           # short, the map is the story
SUBTITLE = "US data centers today, and the AI campuses being built next"
ABOUT    = ("Grey dots are existing data centers mapped in OpenStreetMap. Circles are the large AI campuses "
            "tracked by Epoch AI, sized by the power each will draw when fully built: amber ones are already "
            "running, glowing pink ones are under construction or expanding. A few campuses with no public "
            "street address are placed at their town.")
CREDITS  = ["im3", "epoch"]                  # data sources, see mapkit/style.py (CREDITS)

REGION   = "us48"     # "us48" | a metro: "nyc" "philly" "dc_nova" "sf_bay" "dallas" "austin" "seattle" "quincy_wa" | states: ["TX", "WA"]
TOOL     = "mapbox"   # "mapbox" (interactive) | "folium" (interactive) | "static" (PNG only)
MAPBOX_STYLE = "dark" # "dark" | "light" | "standard" | "satellite"
PROJECTION   = "globe"  # mapbox only: "globe" | "mercator" | "albers" | "equalEarth"

## 2 · Load the master table
Built once from `data/raw`, then read from `data/processed/datacenters_master.geojson`. Use `ai.master(rebuild=True)` if you download newer raw data.

In [2]:
import pandas as pd
import mapkit as mk
from mapkit import ai, publish

dcs = ai.clip(ai.master(), REGION)
print(f"{(dcs.kind == 'existing').sum():,} existing data centers · {(dcs.kind == 'ai_campus').sum()} AI campuses")
print("already in the master table:", [c for c in dcs.columns if c not in ai.BASE_COLS + ["geometry"]] or "nothing extra yet")

1,472 existing data centers · 77 AI campuses
already in the master table: nothing extra yet


c:\Users\gauta\.conda\envs\geospatial\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


## 3 · Analysis + today's new layer
The part that changes. Pattern for a new map:
```python
water = ai.load_layer("some_file.shp")                       # 1. load today's overlay
dcs = ai.attach(dcs, water, ["column"])                      # 2. give each data center its value
ai.update_master(dcs, ["column"], about="...", source="...", day=DAY)   # 3. save it for later maps
```
Other helpers: `ai.count_near`, `ai.distance_to_nearest`, `ai.hexbin`, `ai.count_in`.

In [3]:
ex  = dcs[dcs.kind == "existing"]
big = dcs[dcs.kind == "ai_campus"]
top_states = ex["state_abb"].value_counts().head(5)
status     = big["status"].value_counts()
power_now, power_peak = big["power_now_mw"].sum(), big["power_peak_mw"].sum()

print("Top states by data centers:", ", ".join(f"{s} {n}" for s, n in top_states.items()))
print("AI campuses by status:", ", ".join(f"{s} {n}" for s, n in status.items()))
if power_now:
    print(f"AI campus power today: {power_now:,.0f} MW  ->  at full build: {power_peak:,.0f} MW ({power_peak / power_now:.1f}x)")

overlays = []          # today's new layer(s) go here, e.g. mk.Layer(water, "Water stress", column="bws", ...)
extra_popup = {}       # master columns to show in hover cards, e.g. {"bws_label": "Water stress"}

Top states by data centers: VA 319, TX 127, CA 112, OR 109, OH 91
AI campuses by status: Operating 34, Expanding 26, Under construction 17
AI campus power today: 12,580 MW  ->  at full build: 43,403 MW (3.5x)


## 4 · Map

In [4]:
layers = overlays + ai.base_layers(dcs, extra_popup=extra_popup)    # overlays underneath, the cast on top
bounds = ai.REGIONS[REGION] if isinstance(REGION, str) else None
credits = mk.style.credits(*CREDITS)

if TOOL == "mapbox":
    m = mk.mapbox(layers, title=TITLE, subtitle=SUBTITLE, about=ABOUT, sources=credits, series="ai",
                  bounds=bounds, style=MAPBOX_STYLE, projection=PROJECTION)
elif TOOL == "folium":
    m = mk.interactive(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                       series="ai", bounds=bounds)
else:
    m = mk.static(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                  series="ai", bounds=bounds, crs="EPSG:5070")
m if TOOL == "folium" else None

## 5 · Export
Writes the map to `output/`, a thumbnail to `png_thumbnail/`, starts a caption in `captions/`, updates the gallery. Then commit + push in GitHub Desktop.

In [5]:
publish.save(m, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, series="ai", study=STUDY,
             tool={"mapbox": "Python + Mapbox", "folium": "Python + Folium", "static": "Python"}[TOOL],
             notebook=NOTEBOOK, credits=mk.style.credits(*CREDITS, full=True))

# ArcGIS Pro day? uncomment:
# publish.to_arcgis({"datacenters": dcs}, day=DAY)

map        output/day01_points-ai.html
thumbnail  png_thumbnail/day01_points-ai.png
caption    captions/day01_points-ai.md
live link  https://gautamik01.github.io/30-day-map-challenge-2026/output/day01_points-ai.html  (after you push)
